In [22]:
import requests
from lxml import etree
import pandas as pd

# Cấu hình
INDEX_URL = "https://www.24h.com.vn/sitemap-index.xml"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
NS = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

# Giới hạn số lượng sitemap con bạn muốn quét
SITEMAP_LIMIT = 20 

def crawl_all_urls_from_sitemaps():
    all_articles = []
    
    try:
        print(f"--- Bước 1: Đang lấy danh sách sitemap con từ Index ---")
        res_index = requests.get(INDEX_URL, headers=HEADERS, timeout=15)
        res_index.raise_for_status()
        root_index = etree.fromstring(res_index.content)
        
        # Lấy danh sách toàn bộ sitemap con
        sub_sitemaps = root_index.xpath("//ns:sitemap/ns:loc/text()", namespaces=NS)
        
        # Giới hạn 20 sitemap theo yêu cầu của bạn
        target_sitemaps = sub_sitemaps[:SITEMAP_LIMIT]
        print(f"Sẽ quét qua {len(target_sitemaps)} sitemap con...")

        # --- Bước 2: Duyệt qua từng sitemap con để lấy link chi tiết ---
        for i, sub_url in enumerate(target_sitemaps, 1):
            try:
                print(f"[{i}/{SITEMAP_LIMIT}] Đang tải: {sub_url}")
                res_sub = requests.get(sub_url, headers=HEADERS, timeout=15)
                res_sub.raise_for_status()
                
                root_sub = etree.fromstring(res_sub.content)
                # Lấy tất cả link bài viết trong thẻ <url>
                links = root_sub.xpath("//ns:url/ns:loc/text()", namespaces=NS)
                
                all_articles.extend(links)
                print(f"   -> Tìm thấy thêm {len(links)} link.")
                
            except Exception as e:
                print(f"   -> Lỗi tại sitemap {sub_url}: {e}")
                continue

        # --- Bước 3: Xuất dữ liệu ra Excel ---
        if all_articles:
            print(f"\n--- Bước 3: Đang xuất {len(all_articles)} link ra Excel ---")
            df = pd.DataFrame(all_articles, columns=['URL_Bai_Viet'])
            
            file_name = f"24h_full_links_{SITEMAP_LIMIT}_sitemaps.xlsx"
            
            # Lưu file (sử dụng định dạng chuẩn)
            df.to_excel(file_name, index=False, engine='openpyxl')
            print(f"Thành công! File đã được lưu với tên: {file_name}")
        else:
            print("Không tìm thấy link nào.")

    except Exception as e:
        print(f"Lỗi hệ thống: {e}")

if __name__ == "__main__":
    crawl_all_urls_from_sitemaps()

--- Bước 1: Đang lấy danh sách sitemap con từ Index ---
Sẽ quét qua 20 sitemap con...
[1/20] Đang tải: https://www.24h.com.vn/sitemap-article-daily.xml
   -> Tìm thấy thêm 227 link.
[2/20] Đang tải: https://www.24h.com.vn/sitemap-category.xml
   -> Tìm thấy thêm 480 link.
[3/20] Đang tải: https://www.24h.com.vn/sitemap-event.xml
   -> Tìm thấy thêm 6774 link.
[4/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-05-0.xml
   -> Tìm thấy thêm 318 link.
[5/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-04-0.xml
   -> Tìm thấy thêm 1868 link.
[6/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-04-1.xml
   -> Tìm thấy thêm 1875 link.
[7/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-04-2.xml
   -> Tìm thấy thêm 1874 link.
[8/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-04-3.xml
   -> Tìm thấy thêm 1790 link.
[9/20] Đang tải: https://www.24h.com.vn/sitemap-article-2026-03-0.xml
   -> Tìm thấy thêm 1849 link.
[10/20] Đang tải: https://www.24h